In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import re



docs = [
    "The new research paper on data science and machine learning is fascinating.",
    "Artificial intelligence is a branch of computer science.",
    "Data analysis helps in making informed business decisions.",
    "Machine learning algorithms are used for predictive analysis.",
    "Natural language processing is a key part of AI and data science.",
    "Deep learning is a subset of machine learning.",
    "Scientists discover a new star using advanced data analysis.",
    "Computer vision is another field of artificial intelligence.",
    "Big data technologies are revolutionizing the industry.",
    "Ethical considerations in AI and data science are crucial.",
    "Predictive analysis is a powerful tool for business.",
    "Machine learning models are used for complex data analysis tasks.",
    "Cloud computing provides scalable infrastructure for data processing.",
    "Cybersecurity is a growing concern for all businesses.",
    "Neural networks are the foundation of deep learning.",
    "The stock market analysis requires real-time data.",
    "Robotics and AI are transforming manufacturing.",
    "Data visualization is key for understanding complex data.",
    "Software development teams use agile methodologies.",
    "The future of technology depends on ethical AI."
]

# Get English stopwords
stop_words = set(stopwords.words('english'))

def filtering_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize and remove stopwords
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

# Apply preprocessing to all documents
preprocessed_docs = [filtering_text(doc) for doc in docs]
print("--- Preprocessed Document Example ---")
print(preprocessed_docs[0])
print("*******************************************************************")


from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_docs)

# Get feature names and create a DataFrame
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

def get_top_tfidf_words(doc_id, df, n=10):
    doc_scores = df.iloc[doc_id].sort_values(ascending=False)
    top_words = doc_scores.head(n)
    return list(top_words.index)

print("--- Top 10 TF-IDF Words---")
print(get_top_tfidf_words(0, tfidf_df))
print("****************************************************************************")


from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Tokenize preprocessed documents for Word2Vec training
tokenized_docs = [doc.split() for doc in preprocessed_docs]

# Train the Word2Vec model
model = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=1, workers=4)

# Find most similar words
print("--- Most similar words to 'data' ---")
print(model.wv.most_similar('data', topn=5))
print("\n--- Most similar words to 'analysis' ---")
print(model.wv.most_similar('analysis', topn=5))
print("\n--- Most similar words to 'science' ---")
print(model.wv.most_similar('science', topn=5))
print("----------------------------------------")

# Prepare data for visualization
vis = ['data', 'analysis', 'machine', 'learning', 'science', 'business', 'ai', 'computer', 'cloud', 'ethical', 'predictive']
word_vectors = [model.wv[word] for word in vis]

# Reduce dimensionality using PCA
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(word_vectors)

# Visualize embeddings
plt.figure(figsize=(10, 8))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1])
for i, word in enumerate(vis):
    plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]))
plt.title('Word2Vec Embeddings (PCA)')
plt.show()


from gensim.corpora import Dictionary
from gensim.models import LdaModel
import numpy as np

# Create a dictionary
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

# Train the LDA model
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=4, passes=10, random_state=42)

print("--- Top 5 words for each topic ---")
for idx, topic in lda_model.print_topics(num_words=5):
    print(f"Topic {idx + 1}: {topic}")
print("***********************************************************************")

topic_assignments = []
for doc_bow in corpus:
    topics = lda_model.get_document_topics(doc_bow)
    # highest probability
    main_topic = max(topics, key=lambda x: x[1])
    topic_assignments.append(main_topic[0] + 1)

doc_topic_df = pd.DataFrame({'Document': range(1, len(docs) + 1), 'Assigned Topic': topic_assignments})
print("\n--- Document Topic Assignments ---")
print(doc_topic_df)
print("**************************************************************************************")